# *Capsicum annuum* ve kapsaisin biyosentezi: keşif amaçlı deneme

> **Not:** Bu notebook bir bilimsel sonuç veya Samandağ örneğine ait doğrulanmış analiz sunmaz. NCBI referans genomu üzerinde biyoinformatik adımlarını denemek ve öğrenmek amacıyla hazırlanmıştır. Otomatik filtreleme ve hizalama çıktıları aday/ara sonuçlardır; deneysel doğrulama yapılmamıştır.

## 1. Gerekli araçların kurulumu


In [ ]:
# Biopython kurulumu ve NCBI resmi veri indirme araçlarının (datasets CLI) indirilmesi
!pip install -q biopython
!wget -q https://ftp.ncbi.nlm.nih.gov/pub/datasets/command-line/v2/linux-amd64/datasets -O /usr/local/bin/datasets
!wget -q https://ftp.ncbi.nlm.nih.gov/pub/datasets/command-line/v2/linux-amd64/dataformat -O /usr/local/bin/dataformat
!chmod +x /usr/local/bin/datasets /usr/local/bin/dataformat

# Kurulumu doğrulama
!datasets --version

## 2. *Capsicum annuum* referans genomunu indirme

Bu hücre NCBI RefSeq `GCF_002878395.1` (UCD10Xv1.1) paketini indirir. Bu erişim numarası Samandağ’dan alınmış örnek olarak tanımlanmamıştır; sonuçları Samandağ biberine özgü kabul etmeyin.


In [ ]:
import os
import glob

# Referans genom erişim numarası (Capsicum annuum - UCD10Xv1.1 RefSeq)
ACCESSION = "GCF_002878395.1"
OUTPUT_DIR = "/content/biber_genomu"
os.makedirs(OUTPUT_DIR, exist_ok=True)

print(f"{ACCESSION} referans tam genomu indiriliyor (yaklaşık 1-2 dakika sürebilir)...")

# Tam genom (fna), anotasyon (gff3), protein (faa) ve CDS dosyalarını indirme
!datasets download genome accession {ACCESSION} \
    --include genome,gff3,protein,cds \
    --filename {OUTPUT_DIR}/capsicum_annuum.zip

# İndirilen arşivi açma
!unzip -q -o {OUTPUT_DIR}/capsicum_annuum.zip -d {OUTPUT_DIR}

# İndirilen dosyaların yollarını listeleme
data_path = f"{OUTPUT_DIR}/ncbi_dataset/data/{ACCESSION}"
for file in os.listdir(data_path):
    full_path = os.path.join(data_path, file)
    size_mb = os.path.getsize(full_path) / (1024 * 1024)
    print(f"Dosya: {file} ({size_mb:.2f} MB)")

## 3. Anotasyon açıklamalarında anahtar sözcük araması

Bu bölüm, açıklama metinlerinde kapsaisin yolak terimlerini arayarak aday CDS/protein kayıtlarını seçmeyi dener. Bu yöntem yalnızca metin eşleşmesidir: kayıtların biyolojik rolünü doğrulamaz ve çok sayıda ilgisiz aday üretebilir.


In [ ]:
from Bio import SeqIO
import glob
import os

ACCESSION = "GCF_002878395.1"
VERI_KLASORU = f"/content/biber_genomu/ncbi_dataset/data/{ACCESSION}"
CIKTI_KLASORU = "/content/kapsaisin_genleri"
os.makedirs(CIKTI_KLASORU, exist_ok=True)

# RefSeq anotasyonlarında kapsaisin yolağı enzimlerini yakalayacak anahtar kelimeler
anahtar_kelimeler = [
    "capsaicin", "pun1", "at3",
    "putative aminotransferase", "vanillin aminotransferase",
    "ketoacyl-acp synthase", "3-oxoacyl-[acyl-carrier-protein] synthase",
    "branched-chain-amino-acid aminotransferase",
    "caffeic acid 3-o-methyltransferase",
    "acyl-[acyl-carrier-protein] thioesterase",
    "phenylalanine ammonia-lyase",
    "cinnamate 4-hydroxylase",
    "transcription factor myb31"
]

# 1. CDS (Kodlayan DNA dizileri) dosyasından filtreleme
cds_dosyalari = glob.glob(f"{VERI_KLASORU}/*cds_from_genomic.fna")
if cds_dosyalari:
    secilen_cds = []
    for record in SeqIO.parse(cds_dosyalari[0], "fasta"):
        tanim_kucuk = record.description.lower()
        if any(kelime in tanim_kucuk for kelime in anahtar_kelimeler):
            secilen_cds.append(record)

    cds_cikti = f"{CIKTI_KLASORU}/referans_genom_kapsaisin_CDS.fasta"
    SeqIO.write(secilen_cds, cds_cikti, "fasta")
    print(f"Tam genomdan {len(secilen_cds)} adet ilişkili CDS dizisi ayıklandı -> {cds_cikti}")

# 2. Protein (.faa) dosyasından filtreleme
protein_dosyalari = glob.glob(f"{VERI_KLASORU}/*.faa")
if protein_dosyalari:
    secilen_proteinler = []
    for record in SeqIO.parse(protein_dosyalari[0], "fasta"):
        tanim_kucuk = record.description.lower()
        if any(kelime in tanim_kucuk for kelime in anahtar_kelimeler):
            secilen_proteinler.append(record)

    prot_cikti = f"{CIKTI_KLASORU}/referans_genom_kapsaisin_Protein.faa"
    SeqIO.write(secilen_proteinler, prot_cikti, "fasta")
    print(f"Tam genomdan {len(secilen_proteinler)} adet ilişkili Protein dizisi ayıklandı -> {prot_cikti}")

## 4. Motif ve hizalama ile aday Pun1 kayıtlarını inceleme

Bu bölüm BAHD ailesiyle ilişkilendirilen motifleri ve bir referans proteinle global hizalamayı kullanarak adayları sıralamayı dener. Motif varlığı tek başına enzim işlevini kanıtlamaz. Bu notebook sürümündeki “Pun1 benzerlik (%)” hesabı gerçek yüzde dizi özdeşliği olarak doğrulanmış değildir; yüzde benzerlik sonucu olarak yorumlanmamalıdır.


In [ ]:
from Bio import SeqIO, Entrez
from Bio.Align import PairwiseAligner
import pandas as pd
import re
import os

Entrez.email = "arastirmaci@universite.edu.tr"

GIRIS_DOSYASI = "/content/kapsaisin_genleri/referans_genom_kapsaisin_Protein.faa"
CIKTI_KLASORU = "/content/kapsaisin_genleri/izole_hatay_seti"
os.makedirs(CIKTI_KLASORU, exist_ok=True)

# 1. NCBI'dan referans olarak kullanılan Capsicum annuum Pun1 protein dizisini çekme
print("1. Doğrulanmış aktif Pun1 (AAV66311.1) referans proteini NCBI'dan alınıyor...")
handle = Entrez.efetch(db="protein", id="AAV66311.1", rettype="fasta", retmode="text")
referans_pun1 = SeqIO.read(handle, "fasta")
handle.close()
print(f"   Referans Pun1 yüklendi: {referans_pun1.id} ({len(referans_pun1.seq)} aa)\n")

# 2. 993 proteini okuma ve 'At3g' gibi yalancı pozitiflerden temizleme
tum_proteinler = list(SeqIO.parse(GIRIS_DOSYASI, "fasta"))
print(f"2. Başlangıç protein sayısı: {len(tum_proteinler)}")

# Katalitik motif düzenli ifadeleri (BAHD Açiltransferaz / Kapsaisin Sentaz imzası)
motif_HXXXD = re.compile(r"H[A-Z]{3}D")
motif_DFGWG = re.compile(r"DFGWG|DFGFG")

aligner = PairwiseAligner()
aligner.mode = "global"

pun1_adaylari = []
cekirdek_yolak_adaylari = {}

# Kapsaisin biyosenteziyle ilişkilendirilmiş olası 7 enzim kategorisi (deneme amaçlı)
hedef_enzim_kategorileri = {
    "1_Pun1_Kapsaisin_Sentaz": ["capsaicin synthase", "acyltransferase", "pun1"],
    "2_pAMT_Vanilin_Aminotransferaz": ["gamma-aminobutyrate transaminase", "vanillin aminotransferase", "putative aminotransferase"],
    "3_KasI_Ketoacil_ACP_Sentaz": ["3-oxoacyl-[acyl-carrier-protein] synthase i", "ketoacyl-acp synthase"],
    "4_FatA_Acil_ACP_Tiyoesteraz": ["acyl-[acyl-carrier-protein] thioesterase"],
    "5_BCAT_Dalli_Zincir_Aminotransferaz": ["branched-chain-amino-acid aminotransferase"],
    "6_COMT_Kafeik_Asit_O_Metiltransferaz": ["caffeic acid 3-o-methyltransferase"],
    "7_PAL_Fenilalanin_Amonyak_Liyaz": ["phenylalanine ammonia-lyase"]
}

for rec in tum_proteinler:
    tanim = rec.description.lower()
    seq_str = str(rec.seq).upper()
    uzunluk = len(seq_str)

    # 'at3g', 'gpat3', 'lpat3', 'cat3', 'stat3', 'low quality' gibi yalancı pozitifleri ele
    if any(x in tanim for x in ["at3g", "gpat3", "lpat3", "cat3", "stat3", "vat3", "mat3", "low quality"]):
        continue

    # A) Gerçek Kapsaisin Sentaz (Pun1) Adaylarını Motif + Uzunluk + Hizalama ile Yakala
    if (380 <= uzunluk <= 480) and motif_HXXXD.search(seq_str) and motif_DFGWG.search(seq_str):
        skor = aligner.score(str(referans_pun1.seq), seq_str)
        benzerlik_yuzdesi = (skor / max(len(referans_pun1.seq), uzunluk)) * 100

        if benzerlik_yuzdesi > 55:  # BAHD ailesi içinden Pun1'e en yakın olanlar
            hxxxd_eslesme = motif_HXXXD.search(seq_str)
            dfgwg_eslesme = motif_DFGWG.search(seq_str)
            pun1_adaylari.append({
                "Record": rec,
                "Protein_ID": rec.id,
                "Uzunluk_aa": uzunluk,
                "Pun1_Benzerlik (%)": round(benzerlik_yuzdesi, 2),
                "HXXXD_Motifi": f"{hxxxd_eslesme.group(0)} (Poz: {hxxxd_eslesme.start()+1})",
                "DFGWG_Motifi": f"{dfgwg_eslesme.group(0)} (Poz: {dfgwg_eslesme.start()+1})",
                "Açıklama": rec.description
            })

    # B) Anotasyon adına göre aday yolak kayıtlarını seç (işlev doğrulaması değildir)
    for kategori, kelimeler in hedef_enzim_kategorileri.items():
        if any(k in tanim for k in kelimeler) and "partial" not in tanim:
            # Her kategoriden en yüksek kaliteli / kanonik uzunluktaki tek temsilciyi tut
            if kategori not in cekirdek_yolak_adaylari:
                cekirdek_yolak_adaylari[kategori] = rec

# Pun1 adaylarını benzerlik yüzdesine göre büyükten küçüğe sırala
pun1_adaylari = sorted(pun1_adaylari, key=lambda x: x["Pun1_Benzerlik (%)"], reverse=True)

# En yüksek eşleşmeli Pun1'i çekirdek sete 1 numara olarak ekle
if pun1_adaylari:
    cekirdek_yolak_adaylari["1_Pun1_Kapsaisin_Sentaz"] = pun1_adaylari[0]["Record"]

# 3. Sonuçları Dosyalara Yazdırma
# Dosya 1: İzole edilmiş en güçlü Pun1 (Kapsaisin Sentaz) adayları
en_iyi_pun1_kayitlari = [x["Record"] for x in pun1_adaylari[:5]]
pun1_cikti = f"{CIKTI_KLASORU}/izole_Pun1_Kapsaisin_Sentaz_Adaylari.faa"
SeqIO.write(en_iyi_pun1_kayitlari, pun1_cikti, "fasta")

# Dosya 2: 993 proteinden damıtılmış Çekirdek Samandağ Kapsaisin Yolağı Seti (Her genden 1 kanonik protein)
cekirdek_kayitlar = []
for kat, r in sorted(cekirdek_yolak_adaylari.items()):
    r.description = f"[{kat}] {r.description}"
    cekirdek_kayitlar.append(r)

cekirdek_cikti = f"{CIKTI_KLASORU}/Samandag_Cekirdek_Kapsaisin_Yolagi_7_Protein.faa"
SeqIO.write(cekirdek_kayitlar, cekirdek_cikti, "fasta")

# Tablo raporu
df_pun1 = pd.DataFrame([{k: v for k, v in x.items() if k != "Record"} for x in pun1_adaylari[:10]])
df_pun1.to_csv(f"{CIKTI_KLASORU}/Pun1_Motif_ve_Benzerlik_Raporu.csv", index=False, encoding="utf-8-sig")

print(f"\n3. Aday filtreleme denemesi tamamlandı (sonuçlar doğrulanmış değildir).")
print(f"   -> 993 protein içinden {len(pun1_adaylari)} aday protein tam motif koşulunu sağladı; aktif enzim oldukları doğrulanmadı.")
print(f"   -> En yüksek eşleşmeli ilk 5 Kapsaisin Sentaz adayı kaydedildi: {pun1_cikti}")
print(f"   -> Anotasyon adına göre {len(cekirdek_kayitlar)} aday kategori seçildi (işlev doğrulanmadı): {cekirdek_cikti}\n")
display(df_pun1)

## 5. NCBI’da Türkiye ile ilişkili kayıtları sorgulama

Sorgu sonuçları NCBI kayıt açıklamalarına bağlıdır. Türkiye menşei veya Samandağ kökeni, yalnızca arama terimlerinden kesinleştirilemez; her kayıt ayrıca kontrol edilmelidir.


In [ ]:
# NCBI'da Türkiye ile ilişkili olarak etiketlenen Capsicum annuum kayıtlarını sorgulama (kökeni ayrıca doğrulayın)
sorgu_turkiye = 'Capsicum annuum[Organism] AND (Turkey[Country] OR Turkey[Title] OR Hatay[All Fields] OR Samandag[All Fields] OR Antakya[All Fields])'

handle = Entrez.esearch(db="protein", term=sorgu_turkiye, retmax=50)
tr_sonuc = Entrez.read(handle)
handle.close()

print(f"NCBI Protein veritabanında Türkiye kaynaklı Capsicum annuum kayıt sayısı: {tr_sonuc['Count']}")

if tr_sonuc["IdList"]:
    fetch_handle = Entrez.efetch(db="protein", id=tr_sonuc["IdList"], rettype="fasta", retmode="text")
    tr_proteinler = list(SeqIO.parse(fetch_handle, "fasta"))
    fetch_handle.close()

    tr_dosya = f"{CIKTI_KLASORU}/NCBI_Turkiye_Biber_Proteinleri.faa"
    SeqIO.write(tr_proteinler, tr_dosya, "fasta")
    print(f"Türkiye kaynaklı {len(tr_proteinler)} protein dizisi '{tr_dosya}' dosyasına kaydedildi.\n")

    for p in tr_proteinler[:10]:
        print(f" - {p.id}: {p.description[:90]}... ({len(p.seq)} aa)")

## 6. Sorgu sonucundaki 127 aa kaydı inceleme

Önceden kaydedilmiş çıktıda `WZN32304.1`, Rubisco büyük alt biriminin kısmi dizisi olarak açıklanmıştır. Bu kapsaisin biyosentez enzimi değildir ve Samandağ’a özgü bir bulgu olarak sunulmamalıdır. Aşağıdaki hücre yalnızca 127 aa kayıtlarını göstermeye yönelik bir dosya/çıktı denemesidir.


In [ ]:
from Bio import SeqIO
import glob
import os

ARAMA_KLASORU = "/content/kapsaisin_genleri"
bulunan_kayitlar = []

# Klasördeki ve alt klasörlerdeki tüm .faa / .fasta dosyalarını tara
tum_dosyalar = glob.glob(f"{ARAMA_KLASORU}/**/*.faa", recursive=True) + \
               glob.glob(f"{ARAMA_KLASORU}/**/*.fasta", recursive=True)

seen_ids = set()

for dosya in tum_dosyalar:
    for record in SeqIO.parse(dosya, "fasta"):
        if len(record.seq) == 127 and record.id not in seen_ids:
            seen_ids.add(record.id)
            bulunan_kayitlar.append((dosya, record))

# Eğer önceki hücredeki 'tr_proteinler' veya 'cekirdek_kayitlar' bellekteyse ve 1 elemanlıysa onu da kontrol et
if not bulunan_kayitlar and 'tr_proteinler' in locals() and len(tr_proteinler) == 1:
    bulunan_kayitlar.append(("Bellek (tr_proteinler)", tr_proteinler[0]))

if bulunan_kayitlar:
    for kaynak, rec in bulunan_kayitlar:
        print("=" * 70)
        print(f"KAYNAK DOSYA : {kaynak}")
        print(f"PROTEİN ID   : {rec.id}")
        print(f"AÇIKLAMA     : {rec.description}")
        print(f"UZUNLUK      : {len(rec.seq)} aa")
        print("=" * 70)

        print("\n--- 1. SAF AMİNO ASİT DİZİSİ (Kopyalamaya Uygun) ---")
        print(str(rec.seq))

        print("\n--- 2. FASTA FORMATI ---")
        print(f">{rec.description}\n{rec.seq}")

        # Ayrı bir dosyaya kaydet
        cikti_dosyasi = "/content/127aa_izole_biber_proteini.fasta"
        SeqIO.write([rec], cikti_dosyasi, "fasta")
        print(f"\n[BİLGİ] Dizi '{cikti_dosyasi}' olarak kaydedildi.")
else:
    print("127 aa uzunluğunda dizi bulunamadı. Son oluşan listedeki tüm diziler yazdırılıyor:\n")
    for dosya in tum_dosyalar:
        kayitlar = list(SeqIO.parse(dosya, "fasta"))
        if len(kayitlar) == 1:
            rec = kayitlar[0]
            print(f">{rec.description} ({len(rec.seq)} aa)\n{rec.seq}\n")

## 7. ColabFold Batch girdisi hazırlama denemesi

Bu bölüm belirli başlık desenleriyle protein kayıtlarını bulup CSV/FASTA girdisi hazırlamayı dener. Girdi dosyası üretmek, ColabFold/AlphaFold tahmini çalıştırıldığı veya yapının doğrulandığı anlamına gelmez. UniProt erişimleri ve başlık eşleşmeleri tekrar kontrol edilmelidir.


In [ ]:
import os
import re
import csv
import json
import urllib.request

# 1. Kapsaisin Biyosentez Yolağı Hedef Enzim Tanımları ve Filtre Desenleri
TARGET_ENZYMES = {
    "PAL": {
        "tr_name": "Fenilalanin Amonyak Liyaz",
        "uniprot_id": "B5LAW0",
        "patterns": [r"\bPAL\d*\b", r"phenylalanine\s+ammonia[\s\-]?lyase"],
    },
    "C4H": {
        "tr_name": "Sinamik Asit-4-Hidroksilaz",
        "uniprot_id": "Q9FQY6",
        "patterns": [r"\bC4H\d*\b", r"\bCYP73A\w*\b", r"cinnamate\s+4[\s\-]?monooxygenase", r"cinnamic\s+acid\s+4[\s\-]?hydroxylase"],
    },
    "C3H": {
        "tr_name": "p-Kumarik Asit-3-Hidroksilaz",
        "uniprot_id": "B5LAT7",
        "patterns": [r"\bC3H\d*\b", r"\bCYP98A\w*\b", r"coumarate\s+3[\s\-]?hydroxylase", r"coumaroyl.*3'?[\s\-]?monooxygenase"],
    },
    "CAOMT": {
        "tr_name": "Kafeik Asit O-Metiltransferaz",
        "uniprot_id": "Q9FQY8",
        "patterns": [r"\bCA?OMT\d*\b", r"caffeic\s+acid\s+(?:3[\s\-]?O[\s\-]?)?methyltransferase"],
    },
    "CS_AT3": {
        "tr_name": "Kapsaisin Sentaz / Asil-Transferaz (Pun1/AT3)",
        "uniprot_id": "D2Y3X2",
        "patterns": [r"\bPUN1\b", r"\bAT3\b", r"\bCSY1\b", r"capsaicin\s+synthase", r"acyltransferase\s+pun1"],
    },
}

CODON_TABLE = {
    'ATA':'I','ATC':'I','ATT':'I','ATG':'M','ACA':'T','ACC':'T','ACG':'T','ACT':'T',
    'AAC':'N','AAT':'N','AAA':'K','AAG':'K','AGC':'S','AGT':'S','AGA':'R','AGG':'R',
    'CTA':'L','CTC':'L','CTG':'L','CTT':'L','CCA':'P','CCC':'P','CCG':'P','CCT':'P',
    'CAC':'H','CAT':'H','CAA':'Q','CAG':'Q','CGA':'R','CGC':'R','CGG':'R','CGT':'R',
    'GTA':'V','GTC':'V','GTG':'V','GTT':'V','GCA':'A','GCC':'A','GCG':'A','GCT':'A',
    'GAC':'D','GAT':'D','GAA':'E','GAG':'E','GGA':'G','GGC':'G','GGG':'G','GGT':'G',
    'TCA':'S','TCC':'S','TCG':'S','TCT':'S','TTC':'F','TTT':'F','TTA':'L','TTG':'L',
    'TAC':'Y','TAT':'Y','TAA':'*','TAG':'*','TGC':'C','TGT':'C','TGA':'*','TGG':'W',
}

def is_nucleotide_seq(seq: str) -> bool:
    clean = re.sub(r"[^A-Za-z]", "", seq).upper()
    return bool(clean) and (sum(1 for c in clean if c in "ATGCNU") / len(clean)) > 0.90

def translate_cds(nt_seq: str) -> str:
    seq = re.sub(r"[^A-Za-z]", "", nt_seq).upper().replace("U", "T")
    best_prot = ""
    for frame in range(3):
        sub = seq[frame:]
        aa_list = []
        for i in range(0, len(sub) - 2, 3):
            aa = CODON_TABLE.get(sub[i:i+3], "X")
            if aa == "*":
                break
            aa_list.append(aa)
        cand = "".join(aa_list)
        if len(cand) > len(best_prot):
            best_prot = cand
    return best_prot

def parse_fasta(fasta_path: str):
    records, header, chunks = [], None, []
    with open(fasta_path, "r", encoding="utf-8") as f:
        for line in f:
            line = line.strip()
            if not line:
                continue
            if line.startswith(">"):
                if header is not None:
                    records.append((header, "".join(chunks)))
                header, chunks = line[1:].strip(), []
            else:
                chunks.append(re.sub(r"\s+", "", line))
        if header is not None:
            records.append((header, "".join(chunks)))
    return records

def classify_header(header: str):
    for key, info in TARGET_ENZYMES.items():
        if any(re.search(pat, header, flags=re.IGNORECASE) for pat in info["patterns"]):
            return key
    return None

def save_outputs(extracted, output_dir="samandag_kapsaisin_genleri", prefix="Samandag"):
    os.makedirs(output_dir, exist_ok=True)
    comb_fasta = os.path.join(output_dir, "all_5_capsaicin_enzymes.fasta")
    batch_csv = os.path.join(output_dir, "colabfold_batch_input.csv")

    with open(comb_fasta, "w", encoding="utf-8") as cf, \
         open(batch_csv, "w", newline="", encoding="utf-8") as csvf:
        writer = csv.writer(csvf)
        writer.writerow(["id", "sequence"])

        for key, hits in extracted.items():
            if not hits:
                continue
            single_path = os.path.join(output_dir, f"{prefix}_{key}.fasta")
            with open(single_path, "w", encoding="utf-8") as sf:
                for i, (hdr, seq) in enumerate(hits, 1):
                    job_id = f"{prefix}_{key}" if len(hits) == 1 else f"{prefix}_{key}_{i}"
                    sf.write(f">{job_id} | {hdr}\n{seq}\n")
                    cf.write(f">{job_id} | {hdr}\n{seq}\n")
                    writer.writerow([job_id, seq])
            print(f"[AYRILDI] {key:6s} -> {single_path} ({len(hits)} dizi)")

    print(f"\nColabFold Batch girdi dosyası hazır: {batch_csv}")

# --- ÇALIŞTIRMA ---
# Kendi Samandağ biberi FASTA dosyanız varsa dosya yolunu yazın, yoksa None bırakın:
LOCAL_FASTA_FILE = None  # Örn: "/content/samandag_biber_genler.fasta"

extracted_genes = {k: [] for k in TARGET_ENZYMES}

if LOCAL_FASTA_FILE and os.path.exists(LOCAL_FASTA_FILE):
    for hdr, seq in parse_fasta(LOCAL_FASTA_FILE):
        enz = classify_header(hdr)
        if enz:
            if is_nucleotide_seq(seq):
                seq = translate_cds(seq)
            extracted_genes[enz].append((hdr, seq))
    save_outputs(extracted_genes, prefix="Samandag")
else:
    for key, info in TARGET_ENZYMES.items():
        url = f"https://rest.uniprot.org/uniprotkb/{info['uniprot_id']}.fasta"
        lines = urllib.request.urlopen(url).read().decode("utf-8").splitlines()
        extracted_genes[key].append((lines[0][1:], "".join(lines[1:])))
    save_outputs(extracted_genes, prefix="Capsicum_annuum")